# 06 - Module 3: BRAS (Blockchain Rule Alignment Score)

Computes RAS, DVR, and BRAS for every (model, XAI, dataset) configuration
using the domain rules in `xai_blockchain_framework.rules`.


In [ ]:
import numpy as np
import pandas as pd

from xai_blockchain_framework.config import CONFIG, PATHS, set_seed
from xai_blockchain_framework.metrics.bras import evaluate_bras
from xai_blockchain_framework.rules import elliptic_rules, ethereum_rules
from xai_blockchain_framework.utils import save_csv

set_seed()
ell = np.load(PATHS.data_processed / 'elliptic_splits.npz')
eth = np.load(PATHS.data_processed / 'ethereum_splits.npz')
ell_idx = np.load(PATHS.results_dir / 'xai_eval_indices_elliptic.npy')
eth_idx = np.load(PATHS.results_dir / 'xai_eval_indices_ethereum.npy')


## Evaluate all configurations

In [ ]:
rows = []
for xai in ['shap', 'lime']:
    for model_name in ['randomforest', 'lightgbm']:
        for ds, splits, indices, rule_fn in [
            ('Elliptic', ell, ell_idx, elliptic_rules),
            ('Ethereum', eth, eth_idx, ethereum_rules),
        ]:
            attrs = np.load(PATHS.results_dir / f'{xai}_{model_name}_{ds.lower()}.npy')
            metrics = evaluate_bras(splits['X_test'], attrs, indices, rule_fn, k=5, alpha=0.5)
            rows.append({
                'Type': 'ML', 'Model': model_name.upper(),
                'XAI': xai.upper(), 'Dataset': ds, **metrics,
            })
df = pd.DataFrame(rows).sort_values('BRAS', ascending=False)
print(df.to_string(index=False))
save_csv(df, PATHS.results_dir / 'module3_bras_results.csv')
